# Group Master Total

Notebook này chạy nhiều group độc lập theo cache.

Mục tiêu:
- Mỗi group vẫn export workbook riêng như bình thường.
- Sau mỗi group, lưu cache vào `outputs/group_runs/staging/<GROUP>/`.
- Có thể chỉ rerun 1 group, bỏ qua các group đã có cache.
- Cuối cùng build workbook master chỉ từ cache, không cần chạy lại tất cả.


In [1]:
import sys
from pathlib import Path

project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.rollrate.group_master_cache import (
    build_master_total_from_staging,
    list_staged_groups,
    
    normalize_run_cfg,
    run_selected_groups,
)

GROUP_RUNS = [
    {
        'name': 'POS',
        'data_path': r'//hcm-filesrv.mafc.vn/Risk/PORTFOLIO/Management report/Projection/Data/POS_Parquet_MSCORE_V2',
        'max_mob': 24,
        'target_mobs': [12],
        'group_portfolio_name': 'TOTAL_POS',
        'segment_cols': ['PRODUCT_TYPE', 'RISK_SCORE', 'SALE_CHANNEL', 'GENDER'],
        'loan_min_vintage': '2023-07-01',
        'run_allocation': True,
        'export_group_workbook': True,
        'export_loan_forecast': True,
    },
    # {
    #     'name': 'NTB',
    #     'data_path': r'D:/backup_ổ c/NTB_Parquet_NTBV3_NEWRG',
    #     'max_mob': 24,
    #     'target_mobs': [24],
    #     'loan_base_mode': 'latest_per_loan',
    #     'loan_min_vintage': '2023-07-01',
    #     'group_portfolio_name': 'TOTAL_NTB',
    #     'run_allocation': True,
    #     'export_group_workbook': True,
    #     'export_loan_forecast': True,
    # },
    # {
    #     'name': 'ETB',
    #     'data_path': r'//hcm-filesrv.mafc.vn/Risk/PORTFOLIO/Management report/Projection/Data/ETB_Parquet',
    #     'max_mob': 24,
    #     'target_mobs': [24],
    #     'group_portfolio_name': 'TOTAL_ETB',
    #     'loan_min_vintage': '2023-07-01',
    #     'run_allocation': True,
    #     'export_group_workbook': True,
    #     'export_loan_forecast': True,
    # },
    # Example group with overrides:
    # {
    #     'name': 'POS_C',
    #     'data_path': r'C:\Users\MAFC4709\Python_work\POS_Parquet_FIX23',
    #     'max_mob': 24,
    #     'target_mobs': [12],
    #     'group_portfolio_name': 'TOTAL_POS_C',
    #     'product_filter': ['C'],
    #     'segment_cols': ['PRODUCT_TYPE', 'RISK_SCORE'],
    #     'roll_window': 20,
    #     'decay_lambda': 0.5 ** (1 / 20),
    #     'min_obs': 100,
    #     'min_ead': 1e2,
    #     'k_post_mature': 0.03,
    # },
]

MASTER_PORTFOLIO_NAME = 'TOTAL_ALL_GROUPS'
OUTPUT_ROOT = Path('outputs') / 'group_runs'
STAGING_ROOT = OUTPUT_ROOT / 'staging'

# OUTPUT_ROOT = Path(r"\\hcm-filesrv.mafc.vn\Risk\PORTFOLIO\Management report\Projection\Output")
# STAGING_ROOT = OUTPUT_ROOT / "staging"


# Run control
ACTIVE_GROUPS = None
# Example: ACTIVE_GROUPS = ['POS']

FORCE_GROUPS = []
# Example: FORCE_GROUPS = ['POS']

SKIP_EXISTING_STAGE = True
SAVE_FULL_GROUP_CACHE = False
LOAD_FROM_STAGING_ONLY = False

# Master build control
USE_ALL_STAGED_GROUPS_FOR_MASTER = False
MASTER_INCLUDE_GROUPS = None
STRICT_CONFIG_MATCH_FOR_MASTER = True
# Example: MASTER_INCLUDE_GROUPS = ['POS', 'NTB']

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([
    {
        'name': normalize_run_cfg(cfg)['name'],
        'data_path': normalize_run_cfg(cfg)['data_path'],
        'max_mob': normalize_run_cfg(cfg)['max_mob'],
        'target_mobs': ', '.join(map(str, normalize_run_cfg(cfg)['target_mobs'])) or '-',
        'group_portfolio_name': normalize_run_cfg(cfg)['group_portfolio_name'],
    }
    for cfg in GROUP_RUNS
]))

print('Staged groups:', list_staged_groups(STAGING_ROOT))
print('ACTIVE_GROUPS =', ACTIVE_GROUPS)
print('FORCE_GROUPS =', FORCE_GROUPS)
print('SKIP_EXISTING_STAGE =', SKIP_EXISTING_STAGE)
print('SAVE_FULL_GROUP_CACHE =', SAVE_FULL_GROUP_CACHE)
print('LOAD_FROM_STAGING_ONLY =', LOAD_FROM_STAGING_ONLY)
print('USE_ALL_STAGED_GROUPS_FOR_MASTER =', USE_ALL_STAGED_GROUPS_FOR_MASTER)
print('MASTER_INCLUDE_GROUPS =', MASTER_INCLUDE_GROUPS)
print('STRICT_CONFIG_MATCH_FOR_MASTER =', STRICT_CONFIG_MATCH_FOR_MASTER)


,name,data_path,max_mob,target_mobs,group_portfolio_name
0,POS,//hcm-filesrv.mafc.vn/Risk/PORTFOLIO/Managemen...,24,12,TOTAL_POS


Staged groups: ['NTB']
ACTIVE_GROUPS = None
FORCE_GROUPS = []
SKIP_EXISTING_STAGE = True
SAVE_FULL_GROUP_CACHE = False
LOAD_FROM_STAGING_ONLY = False
USE_ALL_STAGED_GROUPS_FOR_MASTER = False
MASTER_INCLUDE_GROUPS = None
STRICT_CONFIG_MATCH_FOR_MASTER = True


## Run Groups

Cách dùng thường gặp:
- Chạy hết từ đầu: `ACTIVE_GROUPS = None`, `SKIP_EXISTING_STAGE = False`
- Chỉ rerun 1 group: `ACTIVE_GROUPS = ['POS']`, `FORCE_GROUPS = ['POS']`
- Chỉ build master từ cache: `LOAD_FROM_STAGING_ONLY = True`
- Muốn lưu cache đầy đủ hơn để debug / export lại: `SAVE_FULL_GROUP_CACHE = True`


In [2]:
run_summary_df = pd.DataFrame()

if LOAD_FROM_STAGING_ONLY:
    print('Skip run groups. Will use staged data only.')
else:
    run_summary_df = run_selected_groups(
        GROUP_RUNS,
        output_root=OUTPUT_ROOT,
        staging_root=STAGING_ROOT,
        selected_groups=ACTIVE_GROUPS,
        skip_existing_stage=SKIP_EXISTING_STAGE,
        force_groups=FORCE_GROUPS,
        save_full_cache=SAVE_FULL_GROUP_CACHE,
    )
    display(run_summary_df)



RUN GROUP: POS
{
  "name": "POS",
  "data_path": "//hcm-filesrv.mafc.vn/Risk/PORTFOLIO/Management report/Projection/Data/POS_Parquet_MSCORE_V2",
  "max_mob": 24,
  "target_mobs": [
    12
  ],
  "group_portfolio_name": "TOTAL_POS",
  "segment_cols": [
    "PRODUCT_TYPE",
    "RISK_SCORE",
    "SALE_CHANNEL",
    "GENDER"
  ],
  "loan_min_vintage": "2023-07-01",
  "run_allocation": true,
  "export_group_workbook": true,
  "export_loan_forecast": true,
  "product_filter": null,
  "risk_filter": null,
  "loan_base_mode": "latest_cutoff",
  "notes": ""
}
📦 Loading Parquet from: \\hcm-filesrv.mafc.vn\Risk\PORTFOLIO\Management report\Projection\Data\POS_Parquet_MSCORE_V2
✅ Loaded 27,391,955 rows via pyarrow.dataset from \\hcm-filesrv.mafc.vn\Risk\PORTFOLIO\Management report\Projection\Data\POS_Parquet_MSCORE_V2
   ✅ Tạo RISK_SCORE từ ['RISK_SCORE', 'SALE_CHANNEL', 'GENDER']: 46 unique values
📊 Data: 27,391,955 rows | 1,724,093 loans
   SEGMENT_COLS: ['PRODUCT_TYPE', 'RISK_SCORE', 'SALE_CHAN

D:\backup_ổ c\RR_model_1911_Projection\src\rollrate\lifecycle.py:858: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_m.groupby(state_col)[ead_col].sum()
D:\backup_ổ c\RR_model_1911_Projection\src\rollrate\lifecycle.py:858: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_m.groupby(state_col)[ead_col].sum()
D:\backup_ổ c\RR_model_1911_Projection\src\rollrate\lifecycle.py:858: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


✔ Export lifecycle với Config_Info thành công → outputs\group_runs\POS\POS_Lifecycle_20260624_171034.xlsx
📌 Loan export base: latest_cutoff = 2026-05-01
   Filter VINTAGE_DATE >= 2023-07-01: 1,319,754 -> 1,319,754 loans
   Loan export vintage range: 2024-05-01 -> 2026-05-01
🎯 Phân bổ forecast TỐI ƯU tại 1 MOB: [12] (ULTRA FAST)
   ✅ Lấy actual từ df_raw khi có
   ✅ Allocate forecast khi cần

📍 Processing MOB 12
   📊 Extracting actual loans @ MOB 12 (VECTORIZED)...
      Found 950 cohorts with actual data
      ✅ Extracted 1,108,917 actual loans in 124.40s

   📊 Split:
      Actual loans: 1,108,917 (84.0%)
      Need allocation: 613,934 (46.5%)

   🔄 Allocating forecast for 613,934 loans...
📍 Phân bổ forecast tại MOB = 12 (ULTRA FAST mode)
   Số loans: 613,934
   Đang tính combined matrices...
   Cached 536 combined matrices
   Đang tính state probabilities...
   Đang assign states...
   Đang lấy DEL rates từ lifecycle...
   Đang phân bổ EAD theo state (VECTORIZED)...

✅ Phân bổ hoàn tấ

,GROUP,MAX_MOB,TARGET_MOBS,ROWS_RAW,ROWS_GROUP_TOTAL,ALLOCATED_LOANS,GROUP_WORKBOOK,LOAN_WORKBOOK,STAGING_DIR,STATUS
0,POS,24,12,27391955,1025,1319754,outputs\group_runs\POS\POS_Lifecycle_20260624_...,outputs\group_runs\POS\POS_Loan_Forecast_20260...,outputs\group_runs\staging\POS,ran


## Build Master Workbook From Cache

Mặc định phần này sẽ đọc tất cả group đã stage và build ra workbook master.

Nếu chỉ muốn master cho một vài group đã stage:
- `USE_ALL_STAGED_GROUPS_FOR_MASTER = False`
- `MASTER_INCLUDE_GROUPS = ['POS', 'NTB']`


In [3]:
master_result = build_master_total_from_staging(
    GROUP_RUNS,
    MASTER_PORTFOLIO_NAME,
    output_root=OUTPUT_ROOT,
    staging_root=STAGING_ROOT,
    use_all_staged_groups=USE_ALL_STAGED_GROUPS_FOR_MASTER,
    include_groups=MASTER_INCLUDE_GROUPS,
    strict_config_match=STRICT_CONFIG_MATCH_FOR_MASTER,
)

print('Master workbook:', master_result['master_file'])
print('Groups used:', master_result['groups_used'])
display(master_result['summary_df'])
display(master_result['coverage'])


✔ Export lifecycle multi-product thành công → outputs\group_runs\TOTAL_ALL_GROUPS_20260624_172127.xlsx
Master workbook: outputs\group_runs\TOTAL_ALL_GROUPS_20260624_172127.xlsx
Groups used: ['POS']


,GROUP,MAX_MOB,TARGET_MOBS,ROWS_GROUP_TOTAL,STAGING_DIR
0,POS,24,12,1025,outputs\group_runs\staging\POS


,MOB,N_GROUPS
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1
5,5,1
6,6,1
7,7,1
8,8,1
9,9,1
